In [1]:
from pathlib import Path

import pyam
import yaml
import matplotlib.pyplot as plt

from utils import add_gwp100_kyoto, calculate_crossing_timings, extend_flatline_after_netzero

%load_ext autoreload
%autoreload 2

/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/scmdata/database/_database.py:9: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  import tqdm.autonotebook as tqdman
[WARNING] 14:57:52 - pint.util: Redefining 'C' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 14:57:52 - pint.util: Redefining 'N' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 14:57:52 - pint.util: Redefining 'NOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 14:57:52 - pint.util: Redefining 'gNOX' (<class 'pint.delegates.txt_defparser.pla

In [2]:
# Configuration
INPUT_DATA = Path("output/101_harmonised_data.csv")
GHG_MAPPING_FILE = Path("ghg_mapping.yml")
OUTPUT_DATA = Path("output/102_netzero_timings.csv")
OUTPUT_CO2_EXTENDED = Path("output/102_netzero_co2_extended.csv")
OUTPUT_KYOTO_EXTENDED = Path("output/102_netzero_kyoto_extended.csv")

## Step 1: Load Data and Prepare Variables

In [3]:
df = pyam.IamDataFrame(INPUT_DATA)

# Add total CO2 emissions (Fossil + Biosphere)
df.add(
    a="Emissions|CO2|Fossil",
    b="Emissions|CO2|Biosphere",
    name="Emissions|CO2",
    append=True
)

# Load GHG mapping configuration
with open(GHG_MAPPING_FILE) as f:
    ghg_mapping = yaml.safe_load(f)

[INFO] 14:57:53 - pyam.core: Reading file output/101_harmonised_data.csv


## Step 2: Calculate Kyoto Gases (AR6-GWP100)

In [4]:
df_with_kyoto = add_gwp100_kyoto(
    df,
    ghg_mapping["Emissions|Kyoto Gases [AR6-GWP100]"],
    gwp_instance="AR6GWP100"
)

[WARNING] 14:57:59 - pyam.core: Filtered IamDataFrame is empty!
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/scmdata/run.py:2632: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([ret._meta.to_frame(), *to_join_metas]).astype("category")


## Step 3: Calculate Net-Zero Timing for CO2 and Kyoto Gases

In [5]:
# Calculate when CO2 emissions cross zero
calculate_crossing_timings(
    df_with_kyoto,
    variable="Emissions|CO2",
    level=0,
    meta_column="year_netzero_co2"
)

# Calculate when Kyoto gases cross zero
calculate_crossing_timings(
    df_with_kyoto,
    variable="Emissions|Kyoto Gases (AR6-GWP100)",
    level=0,
    meta_column="year_netzero_kyoto_ar6gwp100"
)

# Display scenarios with their net-zero timings
df_with_kyoto.meta[["year_netzero_co2", "year_netzero_kyoto_ar6gwp100"]]

year_netzero_co2  \
model             scenario                                  
AIM/CGE 2.0       SSP1-19                          2058.0   
                  SSP1-26                             NaN   
                  SSP1-34                             NaN   
                  SSP1-45                             NaN   
                  SSP1-Baseline                       NaN   
...                                                   ...   
WITCH-GLOBIOM 4.4 CD-LINKS-NPi                        NaN   
                  CD-LINKS-NPi2020_1000            2079.0   
                  CD-LINKS-NPi2020_1600            2098.0   
                  CD-LINKS-NPi2020_400             2059.0   
                  CD-LINKS-No-Policy                  NaN   

                                         year_netzero_kyoto_ar6gwp100  
model             scenario                                             
AIM/CGE 2.0       SSP1-19                                      2073.0  
                  SSP1-26                                         NaN  
                  SSP1-34                                         NaN  
                  SSP1-45                                         NaN  
                  SSP1-Baseline                                   NaN  
...                                                               ...  
WITCH-GLOBIOM 4.4 CD-LINKS-NPi                                    NaN  
                  CD-LINKS-NPi2020_1000                        2093.0  
                  CD-LINKS-NPi2020_1600                           NaN  
                  CD-LINKS-NPi2020_400                         2071.0  
                  CD-LINKS-No-Policy                              NaN  

[1501 rows x 2 columns]

## Step 4: Mark Scenarios that Achieve Net-Zero

In [6]:
# Identify scenarios that achieve net-zero
yes_co2 = df_with_kyoto.meta.dropna(subset='year_netzero_co2').index
yes_ghg = df_with_kyoto.meta.dropna(subset='year_netzero_kyoto_ar6gwp100').index

# Add boolean metadata flags
df_with_kyoto.set_meta(name='achieve_netzero_co2', meta=False)
df_with_kyoto.set_meta(name='achieve_netzero_co2', meta=True, index=yes_co2)

df_with_kyoto.set_meta(name='achieve_netzero_ghg', meta=False)
df_with_kyoto.set_meta(name='achieve_netzero_ghg', meta=True, index=yes_ghg)

print(f"Scenarios achieving net-zero CO2: {len(yes_co2)}")
print(f"Scenarios achieving net-zero Kyoto gases: {len(yes_ghg)}")

Scenarios achieving net-zero CO2: 743
Scenarios achieving net-zero Kyoto gases: 328


## Step 5: Create Net-Zero CO2 Extension Scenarios

In [7]:
df_co2_extended = extend_flatline_after_netzero(
    df_with_kyoto.filter(achieve_netzero_co2=True),
    constituent_variables=ghg_mapping["Emissions|CO2"],
    aggregate_variable="Emissions|CO2",
    netzero_meta_column="year_netzero_co2",
    scenario_suffix="_NZCO2",
    n_workers=6
)
df_co2_extended.filter(variable=["Emissions|CO2", "Emissions|Kyoto Gases (AR6-GWP100)"], keep=False, inplace=True)

print(f"CO2 extension complete: {len(df_co2_extended.meta)} scenarios")
df_co2_extended.timeseries().to_csv(OUTPUT_CO2_EXTENDED)
print(f"Saved to {OUTPUT_CO2_EXTENDED}")

Processing scenarios: 100%|██████████| 743/743 [00:28<00:00, 26.27it/s]


CO2 extension complete: 743 scenarios
Saved to output/102_netzero_co2_extended.csv


## Step 6: Create Net-Zero Kyoto Gases Extension Scenarios

In [8]:
df_kyoto_extended = extend_flatline_after_netzero(
    df_with_kyoto.filter(achieve_netzero_ghg=True),
    constituent_variables=ghg_mapping["Emissions|Kyoto Gases [AR6-GWP100]"],
    aggregate_variable="Emissions|Kyoto Gases (AR6-GWP100)",
    netzero_meta_column="year_netzero_kyoto_ar6gwp100",
    scenario_suffix="_NZKyoto",
    n_workers=6
)
df_kyoto_extended.filter(variable=["Emissions|CO2", "Emissions|Kyoto Gases (AR6-GWP100)"], keep=False, inplace=True)

print(f"Kyoto extension complete: {len(df_kyoto_extended.meta)} scenarios")
df_kyoto_extended.timeseries().to_csv(OUTPUT_KYOTO_EXTENDED)
print(f"Saved to {OUTPUT_KYOTO_EXTENDED}")

Processing scenarios: 100%|██████████| 328/328 [00:11<00:00, 29.41it/s]


Kyoto extension complete: 328 scenarios
Saved to output/102_netzero_kyoto_extended.csv


## Step 7: Save Complete Dataset with Metadata

In [9]:
# Save complete dataset with net-zero timings as metadata
df_with_kyoto.timeseries().to_csv(OUTPUT_DATA)
print(f"Saved complete dataset to {OUTPUT_DATA}")

# Summary
print(f"\nSummary:")
print(f"  Total scenarios: {len(df_with_kyoto.meta.index)}")
print(f"  Scenarios with CO2 net-zero: {len(yes_co2)} ({len(yes_co2)/len(df_with_kyoto.meta)*100:.1f}%)")
print(f"  Scenarios with Kyoto net-zero: {len(yes_ghg)} ({len(yes_ghg)/len(df_with_kyoto.meta)*100:.1f}%)")
print(f"\nOutput files:")
print(f"  - {OUTPUT_DATA}")
print(f"  - {OUTPUT_CO2_EXTENDED}")
print(f"  - {OUTPUT_KYOTO_EXTENDED}")

Saved complete dataset to output/102_netzero_timings.csv

Summary:
  Total scenarios: 1501
  Scenarios with CO2 net-zero: 743 (49.5%)
  Scenarios with Kyoto net-zero: 328 (21.9%)

Output files:
  - output/102_netzero_timings.csv
  - output/102_netzero_co2_extended.csv
  - output/102_netzero_kyoto_extended.csv
